# Hard Negative Mining - Kaggle

Notebook mineuje hard negatives pre najlepsi dense setup z `new_pipeline.md`:
- model: `intfloat/multilingual-e5-large`
- query mode: `translated`

Ukladanie je nastavene rovnako ako v ostatnych Kaggle notebookoch: vystupy idu do `/kaggle/working/.cache/`.

## Pred spustenim

Pridaj cez **Add Data** tieto datasety:
1. `ct26-translations`
   Ocekavana struktura:
   - `/kaggle/input/ct26-translations/translations/de_train.json`
   - `/kaggle/input/ct26-translations/translations/fr_train.json`
   - volitelne aj `en_train.json`
2. Ak chces mineovat nie cez base model, ale cez lokalny checkpoint, pridaj aj model dataset a zmen `MODEL_NAME` v config bunke.

Dataset collection/train split sa cita priamo z Hugging Face `sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims`.


In [ ]:
# -- 1. Installs -------------------------------------------------------------
!pip -q install sentence-transformers datasets huggingface_hub

In [ ]:
# -- 2. Config ---------------------------------------------------------------
import json
import random
import re
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DATASET_NAME = 'sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims'

# Best dense config from new_pipeline.md
MODEL_NAME = 'intfloat/multilingual-e5-large'
QUERY_MODE = 'translated'

# If you add a Kaggle model dataset, replace MODEL_NAME for example with:
# MODEL_NAME = '/kaggle/input/ct26-e5-finetuned/final'

LANGS = ['en', 'de', 'fr']
TOP_K_NEG = 7
CORPUS_BATCH_SIZE = 64
QUERY_BATCH_SIZE = 64
SEED = 42

CACHE_DIR = Path('/kaggle/working/.cache')
TRANS_BASE = Path('/kaggle/input/ct26-translations')

OUTPUT_NAME = 'hard_negatives_e5_large_translated.jsonl'
OUTPUT_PATH = CACHE_DIR / OUTPUT_NAME

MODEL_SLUG = MODEL_NAME.replace('/', '_').replace('\\', '_').replace(':', '_')
CORPUS_CACHE = CACHE_DIR / f'corpus_embeddings_{MODEL_SLUG}.npy'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

print(f'Model:        {MODEL_NAME}')
print(f'Query mode:   {QUERY_MODE}')
print(f'Output:       {OUTPUT_PATH}')
print(f'Corpus cache: {CORPUS_CACHE}')
print(f'GPU count:    {torch.cuda.device_count()}')
if torch.cuda.is_available():
    print(f'GPU 0:        {torch.cuda.get_device_name(0)}')


In [ ]:
# -- 3. Optional Hugging Face login from Kaggle Secrets ---------------------
import os

try:
    from huggingface_hub import login
    hf_token = os.environ.get('HF_TOKEN', '')
    if hf_token:
        login(token=hf_token)
        print('HF login OK')
    else:
        print('HF_TOKEN not set; using public access only')
except Exception as e:
    print(f'HF login skipped: {e}')


In [ ]:
# -- 4. Helpers --------------------------------------------------------------
def clean_tweet(text: str) -> str:
    text = re.sub(r'@user\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'@(\w+)', r'\1', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_translation_cache(lang: str, split: str) -> dict:
    path = TRANS_BASE / 'translations' / f'{lang}_{split}.json'
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        print(f'[{lang}] loaded translations: {path} ({len(data)} rows)')
        return data
    if lang == 'en':
        print(f'[{lang}] no translation cache; using raw English tweets')
        return {}
    raise FileNotFoundError(
        f'Missing translation cache for {lang}: {path}\n'
        'Add the ct26-translations dataset in Kaggle and keep the translations/ folder structure.'
    )


def build_passage(row) -> str:
    title = row.get('title') or ''
    venue = row.get('venue') or ''
    abstract = row.get('abstract') or ''
    authors = row.get('authors') or ''
    return f'passage: {title}. {venue}. {abstract}. {authors}'


def load_collection_data():
    print('Loading collection...')
    collection = load_dataset(DATASET_NAME, 'collection')['collection']
    passage_texts = [build_passage(row) for row in collection]
    corpus_keys = np.array(collection['pubkey'])
    pubkey_to_idx = {int(k): i for i, k in enumerate(corpus_keys)}
    print(f'Collection size: {len(passage_texts)}')
    return passage_texts, corpus_keys, pubkey_to_idx


def load_train_rows(lang: str, pubkey_to_idx: dict, passage_texts: list[str]):
    ds = load_dataset(DATASET_NAME, lang)['train']
    translations = load_translation_cache(lang, 'train')
    rows = []
    for row in ds:
        idx = str(row['index'])
        query_text = translations.get(idx, row['text'])
        query_text = clean_tweet(query_text)
        true_pubkey = int(row['pubkey'])
        true_idx = pubkey_to_idx[true_pubkey]
        rows.append({
            'anchor': f'query: {query_text}',
            'positive': passage_texts[true_idx],
            'true_idx': true_idx,
            'lang': lang,
        })
    print(f'[{lang}] loaded train rows: {len(rows)}')
    return rows


In [ ]:
# -- 5. Load model + collection ---------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print(f'Loading sentence-transformer model: {MODEL_NAME}')
st_model = SentenceTransformer(MODEL_NAME, device='cpu')
if device == 'cuda':
    st_model = st_model.half()
st_model = st_model.to(device)

passage_texts, corpus_keys, pubkey_to_idx = load_collection_data()

In [ ]:
# -- 6. Encode corpus --------------------------------------------------------
if CORPUS_CACHE.exists():
    print(f'Loading cached corpus embeddings: {CORPUS_CACHE}')
    corpus_embs = np.load(str(CORPUS_CACHE))
else:
    print(f'Encoding corpus: {len(passage_texts)} passages')
    corpus_embs = st_model.encode(
        passage_texts,
        batch_size=CORPUS_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(str(CORPUS_CACHE), corpus_embs)
    print(f'Corpus cache saved: {CORPUS_CACHE}')

print(f'Corpus embeddings shape: {corpus_embs.shape}')

In [ ]:
# -- 7. Mine hard negatives --------------------------------------------------
all_examples = []

for lang in LANGS:
    rows = load_train_rows(lang, pubkey_to_idx, passage_texts)
    print(f'[{lang}] encoding {len(rows)} queries...')
    query_embs = st_model.encode(
        [row['anchor'] for row in rows],
        batch_size=QUERY_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    print(f'[{lang}] computing similarities...')
    sims = query_embs @ corpus_embs.T

    print(f'[{lang}] collecting top-{TOP_K_NEG} hard negatives...')
    written = 0
    for i in tqdm(range(len(rows)), desc=f'mine-{lang}'):
        true_idx = rows[i]['true_idx']
        row_sims = sims[i].copy()
        row_sims[true_idx] = -np.inf
        top_neg_indices = np.argsort(-row_sims)[:TOP_K_NEG]

        all_examples.append({
            'anchor': rows[i]['anchor'],
            'positive': rows[i]['positive'],
            'negatives': [passage_texts[j] for j in top_neg_indices],
            'lang': lang,
            'query_mode': QUERY_MODE,
            'dense_model': MODEL_NAME,
        })
        written += 1

    print(f'[{lang}] written: {written}')

print(f'Total examples: {len(all_examples)}')

In [ ]:
# -- 8. Save dataset ---------------------------------------------------------
random.shuffle(all_examples)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Saved: {OUTPUT_PATH} ({size_mb:.1f} MB)')
print(f'Examples: {len(all_examples)}')
print()
sample = all_examples[0]
print('Preview:')
print(sample['anchor'][:160])
print(sample['negatives'][0][:160])

In [ ]:
# -- 9. Output info ----------------------------------------------------------
cache_files = sorted(str(p) for p in CACHE_DIR.glob('*'))
print('Files in /kaggle/working/.cache:')
for p in cache_files:
    print(' ', p)
print()
print('Dataset file for next training step:')
print(f'  {OUTPUT_PATH}')
print()
print('From Kaggle Output tab, publish or download the .cache artifacts as needed.')